In [2]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path(".")

# 공통 컬럼 스키마
COMMON_COLS = [
    "index",            # 행 인덱스(숫자)
    "sandbox_type",     # ICT / REG_ZONE / FIN / IND_LAW / IND_CASE
    "category",         # 구분/제도/분류
    "title",            # 사례명/서비스명/사업명/제목
    "main_content",     # 서비스 내용/현황/주요내용
    "regulatory_issue", # 규제 내용/현행 규제
    "regulatory_relief",# 특례·허용·규제개정현황
    "conditions",       # 부가조건/주요 부가 조건
    "expected_effect",  # 기대효과
    "achievements"      # 성과/개선결과
]

def normalize_ict():
    """
    ICT_규제_샌드박스.csv  → 공통 스키마로 변환
    예상 원본 컬럼:
      - 번호, 구분, 서비스명, 서비스 내용, 규제, 특례 내용, 부가조건, 기대효과, 개선결과
    """
    src = BASE_DIR / "ICT_규제_샌드박스.csv"
    df = pd.read_csv(src)

    mapping = {
        "번호": "index",
        "구분": "category",
        "서비스명": "title",
        "서비스 내용": "main_content",
        "규제": "regulatory_issue",
        "특례 내용": "regulatory_relief",
        "부가조건": "conditions",
        "기대효과": "expected_effect",
        "개선결과": "achievements",
    }

    # 필요한 컬럼만 선택 후 이름 변경
    df = df[list(mapping.keys())].rename(columns=mapping)
    df["sandbox_type"] = "ICT"

    # 공통 컬럼 순서 맞추기
    df = df[COMMON_COLS]
    out = BASE_DIR / "normalized_ICT_규제_샌드박스.csv"
    df.to_csv(out, index=False, encoding="utf-8-sig")
    return df


def normalize_regzone():
    """
    규제자유특구.csv → 공통 스키마로 변환
    예상 원본 컬럼(필요에 따라 이름 수정 가능):
      - 대제목, 중제목, 소제목, 현황, 허용
      * 규제 이슈는 '현황' 안에 서술되어 있을 수 있으므로 그대로 main_content와 함께 넣거나,
        별도 규제 설명 컬럼이 있으면 그걸 mapping에 지정.
    """
    src = BASE_DIR / "규제자유특구.csv"
    df = pd.read_csv(src)

    # 행 인덱스가 따로 없다고 가정하고 1부터 부여
    df["index"] = range(1, len(df) + 1)

    # category를 대제목 / 중제목 결합으로 구성 (컬럼명이 다르면 아래를 수정)
    if {"대제목", "중제목"}.issubset(df.columns):
        df["category"] = df["대제목"].astype(str) + " / " + df["중제목"].astype(str)
    elif "대제목" in df.columns:
        df["category"] = df["대제목"]
    else:
        df["category"] = ""

    mapping = {
        "소제목": "title",
        "현황": "main_content",
        # 규제 이슈가 따로 없으면 현황 일부로 포함된다고 보고 그대로 둔다.
        # 별도 규제 컬럼이 있으면 아래에 추가:
        # "규제내용": "regulatory_issue",
        "허용": "regulatory_relief",
    }

    # 기본값 초기화
    out_df = pd.DataFrame()
    out_df["index"] = df["index"]
    out_df["sandbox_type"] = "REG_ZONE"
    out_df["category"] = df.get("category", "")

    out_df["title"] = df.get("소제목", "")
    out_df["main_content"] = df.get("현황", "")
    # 규제 이슈가 따로 없으면 main_content에서 요약된다고 보고 비워둔다.
    out_df["regulatory_issue"] = df.get("규제내용", "")
    out_df["regulatory_relief"] = df.get("허용", "")
    out_df["conditions"] = ""          # (없음)
    out_df["expected_effect"] = ""     # (없음)
    out_df["achievements"] = ""        # (없음)

    out_df = out_df[COMMON_COLS]
    out = BASE_DIR / "normalized_규제자유특구.csv"
    out_df.to_csv(out, index=False, encoding="utf-8-sig")
    return out_df


def normalize_fin():
    """
    금융_샌드박스_사례.csv → 공통 스키마로 변환
    예상 원본 컬럼:
      - 지정 제도, 서비스명, 서비스 주요 내용, (규제내용/본문 일부), 규제 특례 내용, 주요 부가 조건 내용
    """
    src = BASE_DIR / "금융_샌드박스_사례.csv"
    df = pd.read_csv(src)

    df["index"] = range(1, len(df) + 1)

    out_df = pd.DataFrame()
    out_df["index"] = df["index"]
    out_df["sandbox_type"] = "FIN"
    out_df["category"] = df.get("지정 제도", "")
    out_df["title"] = df.get("서비스명", "")
    out_df["main_content"] = df.get("서비스 주요 내용", "")
    # 규제 이슈가 '규제내용' 또는 '본문 일부' 같은 이름일 수 있으므로 우선순위로 가져오기
    if "규제내용" in df.columns:
        out_df["regulatory_issue"] = df["규제내용"]
    elif "본문 일부" in df.columns:
        out_df["regulatory_issue"] = df["본문 일부"]
    else:
        out_df["regulatory_issue"] = ""
    out_df["regulatory_relief"] = df.get("규제 특례 내용", "")
    out_df["conditions"] = df.get("주요 부가 조건 내용", "")
    out_df["expected_effect"] = ""
    out_df["achievements"] = ""

    out_df = out_df[COMMON_COLS]
    out = BASE_DIR / "normalized_금융_샌드박스_사례.csv"
    out_df.to_csv(out, index=False, encoding="utf-8-sig")
    return out_df


def normalize_ind_law():
    """
    산업융합_샌드박스_법령.csv → 공통 스키마로 변환
    예상 원본 컬럼:
      - 순번, 국조실분류, 사업명, 규제내용
    """
    src = BASE_DIR / "산업융합_샌드박스_법령.csv"
    df = pd.read_csv(src)

    out_df = pd.DataFrame()
    out_df["index"] = df.get("순번", range(1, len(df) + 1))
    out_df["sandbox_type"] = "IND_LAW"
    out_df["category"] = df.get("국조실분류", "")
    out_df["title"] = df.get("사업명", "")
    # 규제내용 일부를 main_content로도 활용
    out_df["main_content"] = df.get("규제내용", "")
    out_df["regulatory_issue"] = df.get("규제내용", "")
    out_df["regulatory_relief"] = ""
    out_df["conditions"] = ""
    out_df["expected_effect"] = ""
    out_df["achievements"] = ""

    out_df = out_df[COMMON_COLS]
    out = BASE_DIR / "normalized_산업융합_샌드박스_법령.csv"
    out_df.to_csv(out, index=False, encoding="utf-8-sig")
    return out_df


def normalize_ind_case():
    """
    산업융합_샌드박스_사례_124_20251024.xlsx → 공통 스키마로 변환
    예상 원본 컬럼:
      - 번호, 구분(Category), 제목, 주요내용, 규제내용, 규제개정현황(유사), 기대효과, 주요성과
    """
    src = BASE_DIR / "산업융합_샌드박스_사례_124_20251024.xlsx"
    df = pd.read_excel(src)

    out_df = pd.DataFrame()
    out_df["index"] = df.get("번호", range(1, len(df) + 1))
    out_df["sandbox_type"] = "IND_CASE"
    # 구분(Category) 라는 이름일 가능성 높음
    if "구분(Category)" in df.columns:
        out_df["category"] = df["구분(Category)"]
    elif "구분" in df.columns:
        out_df["category"] = df["구분"]
    else:
        out_df["category"] = ""
    out_df["title"] = df.get("제목", "")
    out_df["main_content"] = df.get("주요내용", "")
    out_df["regulatory_issue"] = df.get("규제내용", "")
    out_df["regulatory_relief"] = df.get("규제개정현황(유사)", "")
    out_df["conditions"] = ""
    out_df["expected_effect"] = df.get("기대효과", "")
    out_df["achievements"] = df.get("주요성과", "")

    out_df = out_df[COMMON_COLS]
    out = BASE_DIR / "normalized_산업융합_샌드박스_사례.csv"
    out_df.to_csv(out, index=False, encoding="utf-8-sig")
    return out_df


def main():
    dfs = []
    dfs.append(normalize_ict())
    dfs.append(normalize_regzone())
    dfs.append(normalize_fin())
    dfs.append(normalize_ind_law())
    dfs.append(normalize_ind_case())

    master = pd.concat(dfs, ignore_index=True)
    out_master = BASE_DIR / "sandbox_master_normalized.csv"
    master.to_csv(out_master, index=False, encoding="utf-8-sig")
    print(f"통합 마스터 CSV 저장 완료: {out_master}")


if __name__ == "__main__":
    main()


통합 마스터 CSV 저장 완료: sandbox_master_normalized.csv
